# 04 - Fine-tune SFT-BE checkpoint on fixed AllNLI

Notebook này dùng checkpoint SFT-BE hiện tại làm điểm khởi đầu, rồi fine-tune tiếp trên split fixed `70/15/15`.

In [ ]:
from pathlib import Path
import torch

PROJECT_ROOT = Path('/kaggle/working/similarity_search')
%cd {PROJECT_ROOT}
%pip install -q -r fix/requirements-kaggle.txt
# Cai package goc de import similarity_search.sftbe, va cai package fix de chay trainer moi.
%pip install -q -e .
%pip install -q -e fix

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Checkpoint path

Nếu Kaggle không có `models/sftbe_checkpoint/stage0_final.pt`, upload checkpoint này thành Kaggle Dataset rồi sửa đường dẫn bên dưới.

In [ ]:
SFTBE_CHECKPOINT_PATH = 'models/sftbe_checkpoint/stage0_final.pt'
# Vi du khi upload Kaggle Dataset:
# SFTBE_CHECKPOINT_PATH = '/kaggle/input/sftbe-stage0/stage0_final.pt'

if not Path(SFTBE_CHECKPOINT_PATH).exists():
    raise FileNotFoundError(f'Khong thay checkpoint: {SFTBE_CHECKPOINT_PATH}')

In [ ]:
MAX_RETRIEVAL_QUERIES = 0

!python -m similarity_search_fix.models.train_sftbe \
  --input-dir fix/data/processed/allnli_70_15_15/pair-class \
  --output-dir fix/outputs/sftbe_allnli \
  --model-dir fix/models/sftbe_allnli_70_15_15 \
  --checkpoint-path {SFTBE_CHECKPOINT_PATH} \
  --num-train-epochs 1 \
  --batch-size 64 \
  --eval-batch-size 128 \
  --learning-rate 2e-5 \
  --max-retrieval-queries {MAX_RETRIEVAL_QUERIES}

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open('fix/outputs/sftbe_allnli/metrics.json')), indent=2)[:5000])
display(pd.read_csv('fix/outputs/sftbe_allnli/training_log.csv').tail())

In [ ]:
!zip -qr /kaggle/working/fix_sftbe_outputs.zip fix/models/sftbe_allnli_70_15_15 fix/outputs/sftbe_allnli
print('/kaggle/working/fix_sftbe_outputs.zip')